# InferStack step 1: static decode + CUDA graph, decode speed on the real model

Runtime > Change runtime type > **L4 GPU** (same as the scorecard). Run the cells in order; about 20-30 minutes.

What it measures, per batch size (1, 4, 16) and prompt size (48 and 400 words), after the same prefill:
- **native_current**: today's native decode step (Transformers modules + our installed kernels),
- **static_eager**: the new static-buffer decode step without a graph,
- **graph + torch GDN**: the same step captured once as a CUDA graph and replayed,
- **graph + Triton GDN**: the graph using the repo's grouped GDN Triton decode kernel (default on GPU).

It also checks correctness: at every step the static path's greedy token is compared with today's path (teacher-forced), and the max logit difference is reported. Model-step timing only, not a scored serving number. Bring back the ZIP.

## 1. Verify GPU, clone privately, pin source

In [ ]:
import base64, datetime, getpass, json, os, signal, subprocess, sys, time
from pathlib import Path
PLATFORM = 'colab'
COMMIT = 'ff92334197c839f77c40e499f67164ec81eea17a'
BASE = Path('/content' if PLATFORM == 'colab' else '/kaggle/working')
REPO = BASE / 'inferstack'
assert (3, 11) <= sys.version_info[:2] < (3, 14), 'This profile needs Python 3.11–3.13.'
gpu = subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], text=True)
print(gpu)
if PLATFORM == 'colab':
    # One GPU, BF16-capable (compute capability >= 8.0). A100 preferred; L4 accepted.
    # The label goes into every output path so an L4 run can never be filed as A100.
    GPU_LABEL = 'a100' if 'A100' in gpu else 'l4' if 'L4' in gpu else None
    assert GPU_LABEL, 'Choose an A100 or L4 runtime. T4 has no BF16; use the separate T4 notebook for Kaggle.'
else:
    assert len([line for line in gpu.splitlines() if 'T4' in line]) >= 2, 'Choose Kaggle GPU T4 x2.'
    GPU_LABEL = 't4x2'
print('GPU label:', GPU_LABEL)

def clone_pinned():
    if not (REPO / '.git').is_dir():
        if PLATFORM == 'kaggle':
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret('GITHUB_TOKEN').strip()
        else:
            token = getpass.getpass('Read-only GitHub token for this private repo: ').strip()
        env = os.environ.copy()
        env.update(GIT_TERMINAL_PROMPT='0', GIT_CONFIG_COUNT='1',
                   GIT_CONFIG_KEY_0='http.https://github.com/.extraheader',
                   GIT_CONFIG_VALUE_0='AUTHORIZATION: basic '+base64.b64encode(('x-access-token:'+token).encode()).decode())
        result = subprocess.run(['git','clone','-q','https://github.com/sanjoxtech/inferstack.git',str(REPO)],
                                env=env,capture_output=True,text=True)
        del token, env
        if result.returncode:
            raise RuntimeError('Clone failed. Check read-only repository access and Internet connectivity. Credentials were not printed or stored.')
    subprocess.run(['git','checkout','--detach',COMMIT],cwd=REPO,check=True,capture_output=True,text=True)
    actual=subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()
    assert actual == COMMIT
    print('Pinned source:', actual)
clone_pinned()
RUN_ID=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUT=REPO/'results'/f'{PLATFORM}-{GPU_LABEL}-static-decode-{COMMIT[:7]}-{RUN_ID}'
OUT.mkdir(parents=True,exist_ok=False)
(OUT/'gpu-inventory.txt').write_text(gpu)
(OUT/'run-context.json').write_text(json.dumps({'commit':COMMIT,'platform':PLATFORM,'gpu_label':GPU_LABEL,'run_id':RUN_ID},indent=2))

def logged(argv, filename, *, env=None, timeout=3600):
    path=OUT/filename
    start=time.monotonic()
    with path.open('w') as log:
        proc=subprocess.Popen(list(map(str,argv)),cwd=REPO,env=env,stdout=log,stderr=subprocess.STDOUT,start_new_session=True)
        try:
            while proc.poll() is None:
                elapsed=time.monotonic()-start
                if elapsed > timeout:
                    raise TimeoutError(f'Wall-time cap reached: {filename}')
                print(f'{filename}: running, {int(elapsed)} seconds elapsed',flush=True)
                time.sleep(20)
        except BaseException:
            try:
                os.killpg(proc.pid,signal.SIGTERM)
                proc.wait(timeout=20)
            except (ProcessLookupError,subprocess.TimeoutExpired):
                try: os.killpg(proc.pid,signal.SIGKILL)
                except ProcessLookupError: pass
            raise
    print(filename,'exit:',proc.returncode)
    if proc.returncode:
        print(path.read_text(errors='replace')[-3000:])
    return proc.returncode


## 2. Native InferStack environment

In [ ]:
# Reuse the notebook's CUDA-enabled torch; keep changes in a separate virtualenv.
# Colab/Kaggle images may lack the venv package for their current Python (e.g. 3.13):
# install the matching pythonX.Y-venv if creating a virtualenv fails.
def ensure_venv():
    probe=subprocess.run([sys.executable,'-m','venv','--help'],capture_output=True)
    ok=subprocess.run([sys.executable,'-c','import ensurepip'],capture_output=True).returncode==0
    if probe.returncode==0 and ok: return
    assert os.geteuid()==0, 'venv/ensurepip missing and no root to install it'
    pkg=f'python{sys.version_info.major}.{sys.version_info.minor}-venv'
    subprocess.run(['apt-get','update','-qq'],check=True)
    subprocess.run(['apt-get','install','-y','-qq',pkg],check=True)
ensure_venv()
VENV=Path('/tmp/inferstack-native-env')
subprocess.run([sys.executable,'-m','venv','--system-site-packages',str(VENV)],check=True)
PY=VENV/'bin/python'
ENV={**os.environ,'PATH':str(VENV/'bin')+os.pathsep+os.environ['PATH'],
     'PYTHONPATH':str(REPO/'src'),'INFERSTACK_PROVIDER':'colab','CUDA_VISIBLE_DEVICES':'0'}
assert logged([PY,'-m','pip','install','--require-hashes','-r','requirements-dev.lock'],
              'install-core.log',env=ENV,timeout=2700)==0
assert logged([PY,'-m','pip','install','--no-cache-dir','-e','.[dev,native,hf,postgres]',
               'transformers[kernels]==5.22.2'], 'install-native.log',env=ENV,timeout=2700)==0
preflight="""import json, torch, triton, transformers
assert torch.cuda.is_available() and torch.cuda.is_bf16_supported()
print(json.dumps({'torch':torch.__version__,'cuda':torch.version.cuda,'transformers':transformers.__version__,
                  'triton':triton.__version__,'gpu':torch.cuda.get_device_name(0)},indent=2))
"""
assert logged([PY,'-c',preflight],'preflight-native.log',env=ENV,timeout=180)==0
print((OUT/'preflight-native.log').read_text())
(OUT/'pip-freeze-native.txt').write_text(subprocess.check_output([str(PY),'-m','pip','freeze'],env=ENV,text=True))


## 3. Decode benchmark (both native profiles)

In [ ]:
PROFILES_B = {'native_unfused': 'configs/profiles/production-v1.6.0-baseline.toml',
              'native_full': 'configs/profiles/production-v1.6.0.toml'}
def gpu_used_mib():
    return int(subprocess.check_output(['nvidia-smi','--query-gpu=memory.used','--format=csv,noheader,nounits'],text=True).split()[0])
RESULTS = {}
for prof, path in PROFILES_B.items():
    assert gpu_used_mib() <= 1024, f'GPU busy ({gpu_used_mib()} MiB); Runtime > Restart session and re-run from cell 1'
    name = f'static-decode-{prof}'
    RESULTS[name] = logged([PY,'-u','scripts/static_decode_bench.py','--profile',path,
                            '--out',str((OUT/f'{name}.json').relative_to(REPO))], f'{name}.log', env=ENV, timeout=2*60*60)
    print(name, 'OK' if RESULTS[name] == 0 else f'FAILED (exit {RESULTS[name]}); see {name}.log')
print(RESULTS)

## 4. Summary

In [ ]:
lines = ['| profile | prompt words | batch | native now ms/step | static eager | graph + torch GDN | graph + Triton GDN | speedup (graph vs now) | greedy agreement | max logit diff |',
         '|---|---|---|---|---|---|---|---|---|---|']
for prof in PROFILES_B:
    p = OUT/f'static-decode-{prof}.json'
    if not p.exists(): lines.append(f'| {prof} | missing: see static-decode-{prof}.log |'); continue
    d = json.loads(p.read_text())
    for r in d['rows']:
        if 'error' in r: lines.append(f"| {prof} | {r['prompt_words']} | {r['batch']} | error: {r['error'][:80]} |"); continue
        g = r['static_graph']
        lines.append(f"| {prof} | {r['prompt_words']} | {r['batch']} | {r['native_current']['ms_per_step_median']:.1f} | "
                     f"{r['static_eager']['ms_per_step_median']:.1f} | {r['static_graph_torch_gdn']['ms_per_step_median']:.1f} | {g['ms_per_step_median']:.1f} | {r['speedup_graph_vs_current']:.2f}x | "
                     f"{g['greedy_agreement']:.3f} | {g['max_abs_logit_diff']:.3f} |")
table = '\n'.join(lines)
(OUT/'static-decode-summary.md').write_text(table + '\n')
print(table)

## 5. Download evidence

In [ ]:
import shutil
archive_base=BASE/f'inferstack-{PLATFORM}-{GPU_LABEL}-static-decode-{RUN_ID}'
zip_path=shutil.make_archive(str(archive_base),'zip',root_dir=OUT.parent,base_dir=OUT.name)
print('Evidence archive:',zip_path)
if PLATFORM=='colab':
    from google.colab import files
    files.download(zip_path)
else:
    from IPython.display import FileLink, display
    display(FileLink(zip_path))
    print('You can also download the ZIP from the notebook file browser under /kaggle/working.')
